In [2]:
import pandas as pd
from pypdf import PdfReader
import os
import re
import json

In [3]:
root_filepath = '../data/TR_PDFs'
pdf_names = os.listdir(root_filepath)
pdf_paths = [os.path.join(root_filepath, pdf_name) for pdf_name in pdf_names]

In [4]:
def get_units_of_competencies_and_description(pdf_path):
    """
    Given a Training Regulation PDF from TESDA,
    return a list of (code, competency) tuples.
    """
    reader = PdfReader(pdf_path)

    full_text = "\n".join(
        page.extract_text() or ""
        for page in reader.pages[:10]
    )

    codes, descriptions = [], []

    for line in full_text.splitlines():
        match = re.match(
            r"((\w\s*){3}(\d\s*){6})\b\s*(.+)$",
            line
        )

        if match:
            code = re.sub(r"\s+", "", match.group(1))[:9]
            codes.append(code)
            
            description = match.group(4).strip()
            descriptions.append(description)

    return codes, descriptions

In [5]:
# open the json that has the training_regulations and its data id
tr_competencies_filename = "../data/tr_competencies.json"

if not os.path.exists(tr_competencies_filename):
#if True:
    with open('../data/tesda_training_regulations.json', 'r', encoding='utf-8') as file:
        training_regulations = json.load(file)
    # switch the key and value pair
    training_regulations = {
        data_id: training_regulation
        for training_regulation, data_id in training_regulations.items()
    }

    tr_competencies = {
        'Training Regulation': [],
        'Data ID': [],
        'Competency List': [],
        'Competency Description': []
    }

    base_root = '../data/TR_PDFs'
    for file in sorted(os.listdir(base_root)):
        print(file)

        # get data id and name
        data_id = int(re.search(r'#(\d*).pdf', file).group(1))
        #print(data_id, file)
        training_regulation_name = training_regulations[data_id]

        # get the competencies
        filepath = os.path.join(base_root, file)
        codes, descrption = get_units_of_competencies_and_description(filepath)

        # append the bunch
        tr_competencies['Training Regulation'].append(training_regulation_name)
        tr_competencies['Data ID'].append(data_id)
        tr_competencies['Competency List'].append(codes)
        tr_competencies['Competency Description'].append(descrption)

        with open(tr_competencies_filename, "w") as file:
            json.dump(tr_competencies, file)
else:
    with open(tr_competencies_filename, 'r', encoding='utf-8') as file:
        tr_competencies = json.load(file)

TR - [2D Animation NC III] - #123.pdf
TR - [2D Game Art Development NC III] - #464.pdf
TR - [3D Animation NC III] - #251.pdf
TR - [3D Game Art Development NC III] - #477.pdf
TR - [5-Axis CNC Machine Operation NC III] - #1901.pdf
TR - [Able Seafarer Deck NC II (II-5)] - #727.pdf
TR - [Able Seafarer Engine NC II (III-5)] - #728.pdf
TR - [Agricultural Crops Production NC III] - #115.pdf
TR - [Agricultural Crops Production NC II] - #766.pdf
TR - [Agricultural Crops Production NC I] - #49.pdf
TR - [Agricultural Machinery Operation NC II] - #1787.pdf
TR - [Agricultural Machinery Servicing (4-Wheel Tractor) NC III] - #1881.pdf
TR - [Agroentrepreneurship NC III] - #1794.pdf
TR - [Agroentrepreneurship NC II] - #1793.pdf
TR - [Agroentrepreneurship NC IV] - #1795.pdf
TR - [Air Duct Servicing NC II] - #210.pdf
TR - [Animal Health Care and Management NC III] - #250.pdf
TR - [Animal Production (Poultry-Chicken) NC II] - #769.pdf
TR - [Animal Production (Ruminants) NC II] - #768.pdf
TR - [Animal Prod

In [12]:
tr_df = pd.DataFrame(tr_competencies)

In [ ]:
# create a dictionary for each row that contains the code and its description
def create_competency_dict(training_regulation):
    competency_dict = dict(zip(
        training_regulation['Competency List'], 
        training_regulation['Competency Description']
    ))
    return competency_dict

tr_df['Competency Dictionary'] = tr_df.apply(create_competency_dict, axis=1)

In [30]:
# create the overall dictionary
overall_competency_dict = {}
for _, training_regulation in tr_df.iterrows():
    overall_competency_dict.update(training_regulation['Competency Dictionary'])

with open('../data/competencies_dictionary', 'w') as file:
    json.dump(overall_competency_dict, file)

In [37]:
df = pd.DataFrame(
    overall_competency_dict.items(),
    columns=['Code', 'Description']
)
df

,Code,Description
0,500311109,Lead workplace communication
1,500311110,Lead small teams
2,500311111,Develop and practice negotiation skills
3,500311112,Solve problems related to work activities
4,500311113,Use mathematical concepts and techniques
...,...,...
1657,LOG333315,
1658,ICT251301,Utilize Software Methodologies
1659,ICT251302,Develop Responsive Web Design
1660,ICT251303,Create Interactive Websites


In [38]:
df.Description.value_counts()

Description
                                                                   127
Apply quality standards                                             12
Manage own performance                                              10
Apply basic first aid                                                7
Provide effective customer service                                   5
                                                                  ... 
Perform start-up, testing and commissioning for commercial air-      1
Troubleshoot and repair commercial air-conditioning unit             1
Service and maintain commercial air-conditioning unit                1
Install commercial air-conditioning unit                             1
Develop Website Backend Systems                                      1
Name: count, Length: 1417, dtype: int64

In [39]:
127/1662

0.07641395908543923